# 05 — Batch Enrich Papers with OpenAlex

This notebook shows a realistic workflow:
1. Load the example paper dataset (with Dimensions export data)
2. For each paper DOI, query OpenAlex to get up-to-date metadata
3. Merge the enriched fields back into the DataFrame
4. Export the enriched dataset to a new CSV

This is the kind of pipeline used in systematic reviews to enrich citation databases.

In [1]:
import requests
import pandas as pd
import time

In [2]:
MAILTO = 'your.email@example.com'

## 1. Load example papers

In [3]:
df = pd.read_csv('../data/example_papers.csv')
print(f'Loaded {len(df)} papers')
df[['DOI', 'Title', 'Times cited', 'Inclusion decision']].head()

Loaded 10 papers


,DOI,Title,Times cited,Inclusion decision
0,10.3390/s23031255,An Efficient Machine Learning-Based Emotional ...,7,include
1,10.1038/s41598-023-35795-0,Machine learning explainability in nasopharyng...,14,include
2,10.1016/j.ymeth.2023.08.008,Machine learning algorithms for prediction of ...,1,include
3,10.1186/s12874-023-01837-4,Machine learning for predicting neurodegenerat...,8,include
4,10.1038/s41598-023-31542-7,Explanatory predictive model for COVID-19 seve...,10,include


## 2. Helper: fetch one paper from OpenAlex

In [4]:
def fetch_openalex_work(doi: str, mailto: str = '') -> dict | None:
    """Fetch an OpenAlex work by DOI. Returns None on failure."""
    url = f'https://api.openalex.org/works/https://doi.org/{doi}'
    try:
        response = requests.get(url, params={'mailto': mailto}, timeout=10)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f'  Error fetching {doi}: {e}')
        return None


def extract_fields(work: dict | None) -> dict:
    """Extract the fields we want to add from the OpenAlex work object."""
    if work is None:
        return {
            'oa_cited_by_count': None,
            'oa_is_oa': None,
            'oa_oa_status': None,
            'oa_concepts': None,
            'oa_first_author_id': None,
            'oa_first_author_orcid': None,
        }

    authorships = work.get('authorships', [])
    first_author_id = None
    first_author_orcid = None
    if authorships and authorships[0].get('author'):
        first_author_id = authorships[0]['author'].get('id', '').split('/')[-1]
        first_author_orcid = authorships[0]['author'].get('orcid')

    top_concepts = ', '.join(
        c['display_name']
        for c in sorted(work.get('concepts', []), key=lambda x: -x.get('score', 0))[:3]
    )

    return {
        'oa_cited_by_count': work.get('cited_by_count'),
        'oa_is_oa': work.get('open_access', {}).get('is_oa'),
        'oa_oa_status': work.get('open_access', {}).get('oa_status'),
        'oa_concepts': top_concepts,
        'oa_first_author_id': first_author_id,
        'oa_first_author_orcid': first_author_orcid,
    }

## 3. Run the batch enrichment loop

In [5]:
enriched_rows = []

for i, row in df.iterrows():
    doi = row['DOI']
    print(f'[{i+1}/{len(df)}] Fetching: {doi}')

    work = fetch_openalex_work(doi, mailto=MAILTO)
    fields = extract_fields(work)
    enriched_rows.append(fields)

    time.sleep(0.2)  # polite rate limiting

print('\nDone!')

[1/10] Fetching: 10.3390/s23031255
[2/10] Fetching: 10.1038/s41598-023-35795-0
[3/10] Fetching: 10.1016/j.ymeth.2023.08.008
[4/10] Fetching: 10.1186/s12874-023-01837-4
[5/10] Fetching: 10.1038/s41598-023-31542-7
[6/10] Fetching: 10.1155/2023/9738123
[7/10] Fetching: 10.1186/s12859-023-05235-x
[8/10] Fetching: 10.1371/journal.pone.0289348
[9/10] Fetching: 10.1007/s11356-023-27387-2
[10/10] Fetching: 10.1007/s11356-023-26779-8

Done!


## 4. Merge and inspect enriched DataFrame

In [6]:
df_enriched = pd.concat([df, pd.DataFrame(enriched_rows)], axis=1)

# Compare Dimensions citation count vs OpenAlex
compare = df_enriched[['Title', 'Times cited', 'oa_cited_by_count', 'oa_oa_status', 'oa_concepts']].copy()
compare['Title'] = compare['Title'].str[:60]
compare

,Title,Times cited,oa_cited_by_count,oa_oa_status,oa_concepts
0,An Efficient Machine Learning-Based Emotional ...,7,25,gold,"Electroencephalography, Computer science, Arti..."
1,Machine learning explainability in nasopharyng...,14,132,gold,"Nasopharyngeal cancer, Machine learning, Propo..."
2,Machine learning algorithms for prediction of ...,1,18,closed,"Entrapment, Machine learning, Niosome"
3,Machine learning for predicting neurodegenerat...,8,22,gold,"Cohort, Medicine, Cohort study"
4,Explanatory predictive model for COVID-19 seve...,10,39,gold,"Cytokine storm, Coronavirus disease 2019 (COVI..."
5,Monitoring Cardiovascular Problems in Heart Pa...,10,48,hybrid,"Computer science, Artificial intelligence, Mac..."
6,A review and comparative study of cancer detec...,11,46,gold,"Artificial intelligence, Computer science, Mac..."
7,Comparison of the data mining and machine lear...,2,20,gold,"Breed, Weaning, Body weight"
8,Modelling and optimisation of electrocoagulati...,10,38,closed,"Adaptive neuro fuzzy inference system, Respons..."
9,Improving the accuracy of air relative humidit...,2,13,closed,"Mean squared error, Hilbert–Huang transform, C..."


In [7]:
# How often do citation counts agree?
df_enriched['cite_diff'] = df_enriched['oa_cited_by_count'] - df_enriched['Times cited']
print('Citation count difference (OpenAlex - Dimensions):')
print(df_enriched['cite_diff'].describe())

Citation count difference (OpenAlex - Dimensions):
count     10.000000
mean      32.600000
std       31.348223
min       11.000000
25%       17.250000
50%       23.000000
75%       33.500000
max      118.000000
Name: cite_diff, dtype: float64


## 5. Export enriched dataset

In [8]:
output_path = '../data/example_papers_enriched.csv'
df_enriched.drop(columns=['cite_diff']).to_csv(output_path, index=False)
print(f'Saved enriched dataset to {output_path}')
print(f'Shape: {df_enriched.shape}')

Saved enriched dataset to ../data/example_papers_enriched.csv
Shape: (10, 20)
